[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lugerlakes/signal_analysis_course/blob/main/Lab2_Estadistica_descriptiva.ipynb)

<div style="border-left:6px solid #1F4E79;padding:10px 16px;background:#F3F7FB">

<span style="font-size:12px;letter-spacing:1px;color:#1F4E79">CURSO DE GUERRA ELECTRÓNICA · ASIGNATURA: ANÁLISIS DE SEÑALES DE RADAR</span>

# Laboratorio 2 · Estadística descriptiva para el analista EW
### Resumir, graficar e informar parámetros de radar con su incertidumbre

<span style="font-size:12px;color:#7a7a7a">MATERIAL DE INSTRUCCIÓN · DATOS SINTÉTICOS · Emisores y bibliotecas ficticios</span>
</div>

| | |
|---|---|
| ⏱️ **Duración estimada** | 3 a 4 horas |
| 📥 **Datos de entrada** | `D1_ESM_captura_emisor_unico.xlsx` (el mismo emisor del Lab 1) |
| 📤 **Producto al final** | Ficha estadística del emisor (Excel) + respuestas de interpretación |

**Objetivos de aprendizaje** — al terminar este laboratorio serás capaz de:

1. Calcular e interpretar **media, mediana, moda, mínimo, máximo, percentiles, rango, desviación estándar, CV e IQR**.
2. Elegir el estadígrafo correcto según la **forma** de la distribución (simétrica, asimétrica, multimodal).
3. Construir e interpretar **histogramas, curvas KDE, boxplots, violines y ECDF**.
4. Calcular e interpretar correctamente un **intervalo de confianza del 95 %** (y saber qué NO significa).
5. Distinguir un **valor atípico** de un **error**, y evitar filtrados que destruyen información.
6. Elaborar la **ficha estadística** de un emisor para el informe.

**Cómo leer este cuaderno**

| Ícono | Significado |
|---|---|
| 📘 | Concepto o teoría breve |
| ▶️ | Celda de código para **ejecutar** (clic en la celda y `Shift + Enter`) |
| ✏️ | **Tu turno**: completa el código o responde |
| 💡 | Consejo práctico |
| ⚠️ | Error frecuente o precaución |
| 🎯 | Conexión con la Guerra Electrónica (ESM / ELINT) |

> Ejecuta las celdas **en orden, de arriba hacia abajo**. Si algo falla de forma extraña: menú **Entorno de ejecución → Reiniciar y ejecutar todo**.

---
## 0. Preparación

▶️ La siguiente celda contiene la **caja de herramientas** del curso: las mismas operaciones de limpieza del Lab 1, ahora empaquetadas en funciones (`cargar_pdw`, `limpiar_pdw`, …). Está plegada para no distraer: puedes abrirla con doble clic para leerla.

In [ ]:
# @title 🧰 Caja de herramientas (Lab 1: carga y limpieza) — ejecutar
# ------------------------------------------------------------------
# A. CARGA Y LIMPIEZA DE PDW (lo que aprendiste a mano en el Lab 1)
# ------------------------------------------------------------------
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 4)

CARPETAS_DATOS = [".", "datos", "/content", "/content/datos", "/content/sample_data",
                  "/content/drive/MyDrive", "/content/drive/MyDrive/datos",
                  "/content/drive/MyDrive/Analisis_de_Señales"]


def buscar_archivo(nombre):
    """Busca 'nombre' en las carpetas habituales (local y Google Colab)."""
    for carpeta in CARPETAS_DATOS:
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    raise FileNotFoundError(
        f"No encontré '{nombre}'. Súbelo con el ícono de carpeta 📁 (panel izquierdo de Colab) "
        f"o copia su ruta exacta en la variable del archivo.")


def estandarizar_columnas(df):
    """Renombra las columnas de la exportación ESM a nombres estándar con unidades."""
    nuevos = {}
    for col in df.columns:
        c = str(col).upper()
        if c.startswith("UNNAMED") or c.strip() == "":
            continue
        if "TOA" in c:
            nuevos[col] = "TOA_us"
        elif "FREQ" in c or "FREC" in c:
            nuevos[col] = "FREQ_MHz"
        elif c.startswith("PW") or "WIDTH" in c:
            nuevos[col] = "PW_us"
        elif "AMP" in c:
            nuevos[col] = "AMP_dBm"
        elif "AOA" in c or "DOA" in c or "BEARING" in c:
            nuevos[col] = "AOA_deg"
        elif "PRI" in c:
            nuevos[col] = "PRI_us"
    columnas_vacias = [c for c in df.columns if c not in nuevos]
    return df.drop(columns=columnas_vacias).rename(columns=nuevos)


def limpiar_pdw(df, verbose=True):
    """Limpieza estándar de PDW: centinelas, vacíos, duplicados y orden por TOA."""
    n0 = len(df)
    df = df.copy()
    claves = [c for c in ["TOA_us", "FREQ_MHz", "PW_us"] if c in df.columns]
    df = df.dropna(subset=claves)
    n1 = len(df)
    malos = (df["FREQ_MHz"] <= 0) | (df["PW_us"] <= 0)      # registros "fantasma" del receptor
    df = df[~malos]
    n2 = len(df)
    df = df.drop_duplicates()
    n3 = len(df)
    df = df.sort_values("TOA_us").reset_index(drop=True)
    df["TOA_s"] = (df["TOA_us"] - df["TOA_us"].iloc[0]) / 1e6
    df["DTOA_us"] = df["TOA_us"].diff()
    if verbose:
        print(f"Filas originales ............ {n0:>7,}")
        print(f"  − con celdas vacías ....... {n0 - n1:>7,}")
        print(f"  − con ceros / centinela ... {n1 - n2:>7,}")
        print(f"  − duplicadas .............. {n2 - n3:>7,}")
        print(f"Filas válidas ............... {n3:>7,}")
    return df


# Carpeta web con los datos del curso (repositorio público). Déjala vacía ("") para trabajar sólo con archivos locales.
URL_DATOS = "https://raw.githubusercontent.com/lugerlakes/signal_analysis_course/main/datos/"


def asegurar_archivo(nombre):
    """Devuelve la ruta del archivo; si no está, lo descarga (URL_DATOS) o pide subirlo (Colab)."""
    try:
        return buscar_archivo(nombre)
    except FileNotFoundError:
        pass
    if URL_DATOS:
        import urllib.request, urllib.parse
        os.makedirs("datos", exist_ok=True)
        destino = os.path.join("datos", nombre)
        try:
            urllib.request.urlretrieve(URL_DATOS.rstrip("/") + "/" + urllib.parse.quote(nombre), destino)
            print(f"⬇️ '{nombre}' descargado desde el repositorio del curso")
            return destino
        except Exception as e:
            print(f"⚠️ No se pudo descargar '{nombre}' ({e}). Intentando otras opciones…")
            if os.path.exists(destino):
                os.remove(destino)
    try:
        from google.colab import files
        print(f"📁 No encuentro '{nombre}'. Selecciónalo desde tu computador:")
        subidos = files.upload()
        if nombre in subidos:
            return nombre
    except ImportError:
        pass
    return buscar_archivo(nombre)   # mensaje de error claro


def cargar_pdw(nombre_archivo, hoja=0, verbose=True):
    """Carga un archivo ESM (Excel o CSV), estandariza columnas y limpia."""
    ruta = asegurar_archivo(nombre_archivo)
    if ruta.lower().endswith(".csv"):
        crudo = pd.read_csv(ruta)
    else:
        crudo = pd.read_excel(ruta, sheet_name=hoja)
    df = limpiar_pdw(estandarizar_columnas(crudo), verbose=verbose)
    if verbose:
        dur = df["TOA_s"].iloc[-1]
        print(f"Duración de la captura ...... {dur:,.2f} s")
    return df
print('Caja de herramientas lista ✔')

In [ ]:
df = cargar_pdw("D1_ESM_captura_emisor_unico.xlsx")
df.head()

---
## 1. ¿Por qué estadística en Guerra Electrónica?

📘 Cada pulso medido por el receptor es:

$$\text{valor medido} = \text{valor real del emisor} + \text{error de medición}$$

El error depende de la **relación señal/ruido** (pulsos débiles → peor medición), de la resolución del receptor y de su calibración. Además, el propio emisor puede **variar a propósito** sus parámetros (jitter, stagger, agilidad).

Por eso un informe ELINT **nunca** dice "la RF es 2950,92 MHz" (¡eso es un solo pulso!). Dice algo como:

> *RF central 2950,0 MHz (IC 95 %: 2949,96 – 2950,02 MHz), dispersión 0,7 MHz, rango observado P1–P99: 2948,1 – 2952,0 MHz.*

| Concepto | En nuestro contexto |
|---|---|
| **Población** | Todos los pulsos que el radar emite (infinitos, inaccesibles) |
| **Parámetro** | El valor "real" del emisor (p. ej. su RF verdadera) |
| **Muestra** | Los pulsos que **interceptamos** |
| **Estadígrafo** | Lo que calculamos con la muestra (media, mediana, …) para **estimar** el parámetro |

---
## 2. Medidas de tendencia central: ¿cuál es el valor "típico"?

| Estadígrafo | Definición | Fortaleza | Debilidad |
|---|---|---|---|
| **Media** (promedio) $\bar x = \frac{1}{n}\sum x_i$ | Suma / cantidad | Usa toda la información; base del IC | **Muy sensible** a valores extremos |
| **Mediana** | Valor central al ordenar (50 % arriba, 50 % abajo) | **Robusta** ante extremos | Ignora la magnitud de los extremos |
| **Moda** | Valor más frecuente | Indica el **peak** de la distribución; detecta varios modos | En datos continuos hay que agrupar (histograma) |

### 2.1 Ejemplo a mano: un pulso espurio
Nueve PW medidos (µs) de un mismo emisor, más **un pulso espurio** de 6,0 µs que se coló en el grupo:

In [ ]:
pw = pd.Series([1.2, 1.1, 1.2, 1.3, 1.2, 1.1, 1.2, 1.3, 1.2, 6.0])

print(f"Media    : {pw.mean():.2f} µs")
print(f"Mediana  : {pw.median():.2f} µs")
print(f"Moda     : {pw.mode()[0]:.2f} µs")
print()
sin_espurio = pw[pw < 5]
print(f"Sin el espurio → media {sin_espurio.mean():.2f} · mediana {sin_espurio.median():.2f}")

🎯 **Un solo pulso espurio** movió la media de 1,20 a 1,68 µs (+40 %). La mediana no se inmutó. En capturas reales —con ruido, pulsos de otros emisores y pulsos débiles— la **mediana es el valor central más confiable**.

### 2.2 La moda en datos continuos
Con mediciones continuas casi ningún valor se repite exactamente, así que `.mode()` pierde sentido. Se usa la **moda del histograma**: el centro de la barra más alta.

In [ ]:
def moda_continua(serie, bins="auto"):
    """Moda de una variable continua = centro de la barra más alta del histograma."""
    x = pd.Series(serie).dropna().values
    cuentas, bordes = np.histogram(x, bins=bins)
    i = np.argmax(cuentas)
    return 0.5 * (bordes[i] + bordes[i + 1])

for col in ["FREQ_MHz", "PW_us", "AMP_dBm"]:
    s = df[col]
    print(f"{col:9s}  media {s.mean():9.2f}   mediana {s.median():9.2f}   moda {moda_continua(s):9.2f}")

### ✏️ Tu turno 1
Observa los tres resultados:
1. ¿En qué variable media ≈ mediana ≈ moda? ¿Qué te dice eso de su forma?
2. En `PW_us` la media es **menor** que la mediana. ¿Hacia dónde está la "cola"?
3. En `AMP_dBm` las tres difieren mucho. ¿Por qué crees que pasa? (Pista: recuerda el gráfico de amplitud vs tiempo del Lab 1.)

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. En `FREQ_MHz`: distribución **simétrica** (ruido de medición alrededor de la RF real).
2. Cola hacia la **izquierda** (valores bajos): los pulsos débiles se miden más cortos (el umbral corta los flancos del pulso).
3. La amplitud no es "ruido alrededor de un valor": refleja el **patrón de la antena** barriendo. Hay muchos pulsos débiles (lóbulos laterales, ≈ −65 a −72 dBm) y pocos fuertes (lóbulo principal, ≈ −36 dBm). Distribución **asimétrica y con varios modos**: la media no representa nada físico.


</details>

---
## 3. Medidas de posición: mínimo, máximo y percentiles

📘 El **percentil *p*** (P*p*) es el valor bajo el cual queda el *p* % de los datos.

| Nombre | Equivale a | Lectura |
|---|---|---|
| Mínimo / Máximo | P0 / P100 | Valores extremos observados |
| **Q1** (primer cuartil) | P25 | 25 % de los pulsos está por debajo |
| **Mediana** (Q2) | P50 | Mitad y mitad |
| **Q3** (tercer cuartil) | P75 | 75 % por debajo |
| P5 – P95 | — | Rango que contiene el 90 % central de los pulsos |

⚠️ **¿Por qué no reportar mínimo–máximo como "rango del emisor"?** Porque **un solo pulso** (espurio, mal medido o de otro emisor) los define. Los percentiles P1–P99 o P5–P95 son mucho más estables.

In [ ]:
f = df["FREQ_MHz"]
print(f"Mínimo {f.min():.2f}   P1 {f.quantile(0.01):.2f}   P5 {f.quantile(0.05):.2f}   Q1 {f.quantile(0.25):.2f}")
print(f"Mediana {f.median():.2f}")
print(f"Q3 {f.quantile(0.75):.2f}   P95 {f.quantile(0.95):.2f}   P99 {f.quantile(0.99):.2f}   Máximo {f.max():.2f}")

# ¿Qué pasa si se cuela UN pulso de otro emisor a 3100 MHz?
f_contaminada = pd.concat([f, pd.Series([3100.0])])
print(f"\nCon un pulso intruso → máximo {f_contaminada.max():.2f} (¡!)   P99 {f_contaminada.quantile(0.99):.2f} (sin cambio)")

---
## 4. Medidas de dispersión: ¿cuánto varían los datos?

| Estadígrafo | Fórmula / definición | Interpretación EW |
|---|---|---|
| **Rango** | máx − mín | Sensible a un solo pulso extremo |
| **Varianza** $s^2$ | $\frac{1}{n-1}\sum (x_i-\bar x)^2$ | En unidades al cuadrado (poco intuitiva) |
| **Desviación estándar** $s$ | $\sqrt{s^2}$ | Dispersión típica alrededor de la media, **en las mismas unidades** |
| **Coef. de variación** CV | $100 \cdot s / \bar x$ (%) | Dispersión **relativa**: sirve para comparar variables de distinta escala (p. ej. jitter de PRI en %) |
| **IQR** (rango intercuartil) | Q3 − Q1 | Ancho del 50 % central; **robusto** |

🎯 Para un emisor de RF **fija**, la desviación estándar de la RF medida es básicamente la **precisión del receptor**. Si de pronto la RF tiene una dispersión de decenas o cientos de MHz → **agilidad en frecuencia** (o mezcla de emisores).

In [ ]:
for col in ["FREQ_MHz", "PW_us", "AOA_deg"]:
    s = df[col]
    print(f"{col:9s} rango {s.max()-s.min():7.2f}   desv.est {s.std():6.3f}   CV {100*s.std()/s.mean():6.3f} %   IQR {s.quantile(.75)-s.quantile(.25):6.3f}")

### 4.1 La regla 68 – 95 – 99,7
Si los datos siguen una distribución **normal (campana de Gauss)**, aproximadamente:
* 68 % de los valores cae en $\bar x \pm 1s$,
* 95 % en $\bar x \pm 2s$,
* 99,7 % en $\bar x \pm 3s$.

Verifiquémoslo con la RF (que vimos simétrica):

In [ ]:
m, s = f.mean(), f.std()
for k in [1, 2, 3]:
    dentro = f.between(m - k*s, m + k*s).mean() * 100
    print(f"media ± {k}·s  →  {dentro:5.1f} % de los pulsos")

🎯 **No se cumple exactamente:** hay **más** pulsos de lo esperado en el centro (≈ 79 % vs 68 %) **y** más en los extremos (≈ 2 % fuera de ±3s vs 0,3 %). ¿Por qué? Porque la RF medida es una **mezcla** de pulsos fuertes (medidos con mucha precisión) y pulsos débiles (medidos con más error). Ese tipo de mezcla produce una distribución "puntuda con colas pesadas" (*leptocúrtica*).

💡 **La regla 68–95–99,7 sólo vale para datos aproximadamente normales.** No la apliques a la amplitud ni a los DTOA, y úsala con cautela incluso aquí.

---
## 5. `describe()`: el resumen en una línea

In [ ]:
df[["FREQ_MHz", "PW_us", "AMP_dBm", "AOA_deg", "DTOA_us"]].describe().round(2)

| Fila | Significado |
|---|---|
| `count` | nº de valores no vacíos (en `DTOA_us` hay uno menos: el primer pulso no tiene anterior) |
| `mean`, `std` | media y desviación estándar |
| `min`, `max` | extremos |
| `25%`, `50%`, `75%` | Q1, **mediana**, Q3 |

### ✏️ Tu turno 2
En la columna `DTOA_us` la media es ≈ 26 000 µs pero la mediana es ≈ 1100 µs. ¿Cuál de las dos describe la PRI? ¿Por qué la media es tan grande?

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


La **mediana** se acerca a la PRI. La media está inflada por los **huecos entre iluminaciones** (~5,6 s = 5 600 000 µs): bastan 9 valores enormes entre 2000 para multiplicar la media por 25. *Lección: nunca resumas DTOA con la media sin antes separar huecos y pulsos perdidos.*


</details>

---
## 6. Gráficos de distribución

### 6.1 Histograma
📘 Divide el eje en intervalos (*bins*) y cuenta cuántos datos caen en cada uno. La forma depende **mucho** del número de bins:
* muy pocos → se esconden detalles (peaks que se funden);
* demasiados → ruido, barras "peinadas".

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for a, b in zip(ax, [8, 40, 300]):
    sns.histplot(df["FREQ_MHz"], bins=b, ax=a)
    a.set_title(f"RF · {b} bins"); a.set_xlabel("RF (MHz)")
plt.tight_layout(); plt.show()

💡 Prueba varios valores de `bins`. Con 300 bins aparecen "dientes": la RF viene **redondeada a 0,01 MHz** y además el receptor tiene su propia resolución (cuantización).

### 6.2 Histograma de los DTOA: aquí está la PRI
Recortamos a DTOA < 5000 µs para dejar fuera los huecos de scan:

In [ ]:
dtoa = df["DTOA_us"].dropna()
plt.figure(figsize=(13, 3.8))
sns.histplot(dtoa[dtoa < 5000], bins=400)
plt.xlabel("DTOA (µs)"); plt.ylabel("Nº de pulsos")
plt.title("Histograma de DTOA (< 5000 µs)")
plt.show()

🎯 Dos picos altos (≈ 900 y ≈ 1100 µs) de igual altura y peaks menores en ≈ 2000, 2900, 3100, 4000 µs: son **sumas** de 900 y 1100 (pulsos perdidos). Patrón típico de **stagger de 2 posiciones**.

### 6.3 Curva KDE (estimación de densidad por núcleos)
📘 La **KDE** (*Kernel Density Estimation*) es un "histograma suavizado": coloca una pequeña campana sobre cada dato y las suma. El área bajo la curva vale 1. Su parámetro clave es el **ancho de banda** (*bandwidth*): pequeño → curva nerviosa; grande → curva muy lisa que puede **fundir peaks cercanos**.

In [ ]:
sub = dtoa[(dtoa > 700) & (dtoa < 1300)]
fig, ax = plt.subplots(1, 4, figsize=(17, 3.5))
for a, bw in zip(ax, [0.05, 0.5, 2, 8]):
    sns.kdeplot(sub, bw_adjust=bw, ax=a, fill=True)
    a.set_title(f"bw_adjust = {bw}"); a.set_xlabel("DTOA (µs)"); a.set_xlim(600, 1400)
fig.suptitle("Efecto del ancho de banda en la KDE (mismos datos)", y=1.03)
plt.tight_layout(); plt.show()

⚠️ **Precauciones con la KDE**
* Con `bw_adjust` grande (panel derecho), **los dos niveles de PRI se funden en una sola joroba centrada en ≈ 1000 µs** (¡un valor que el radar nunca usa!).
* La KDE puede "desbordar" límites físicos (p. ej. mostrar densidad con PW negativo). En seaborn se evita con `cut=0`.
* El eje Y es **densidad**, no cantidad de pulsos.

### 6.4 Boxplot (diagrama de caja y bigotes)

In [ ]:
x = df["PW_us"]
q1, q2, q3 = x.quantile([0.25, 0.5, 0.75])
iqr = q3 - q1
bigote_inf = x[x >= q1 - 1.5 * iqr].min()
bigote_sup = x[x <= q3 + 1.5 * iqr].max()
n_atip = ((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum()

fig, ax = plt.subplots(figsize=(13, 3.6))
sns.boxplot(x=x, ax=ax, color="#9DB9D9", width=0.35)
for val, txt, y in [(q1, f"Q1 = {q1:.2f}", -0.32), (q2, f"Mediana = {q2:.2f}", 0.36), (q3, f"Q3 = {q3:.2f}", -0.32),
                    (bigote_inf, f"bigote inf.\n{bigote_inf:.2f}", 0.36), (bigote_sup, f"bigote sup.\n{bigote_sup:.2f}", 0.36)]:
    ax.annotate(txt, xy=(val, 0), xytext=(val, y), ha="center", fontsize=9,
                arrowprops=dict(arrowstyle="->", color="gray"))
ax.annotate(f"{n_atip} puntos 'atípicos'\n(fuera de 1,5·IQR)", xy=(x.min(), 0), xytext=(x.min(), -0.4), fontsize=9, color="#C44E52")
ax.set_xlabel("PW (µs)"); ax.set_ylim(0.5, -0.5)
ax.set_title(f"Anatomía de un boxplot · PW del emisor (IQR = {iqr:.2f} µs)")
plt.show()

| Elemento | Qué representa |
|---|---|
| **Caja** | Del Q1 al Q3: el **50 % central** de los pulsos (su ancho es el IQR) |
| **Línea dentro de la caja** | La **mediana** |
| **Bigotes** | Hasta el dato más extremo que esté a menos de **1,5 × IQR** de la caja |
| **Puntos aislados** | Datos más allá de los bigotes: "atípicos" según esta regla |
| Mediana descentrada en la caja / un bigote más largo | **Asimetría** |

⚠️ "Atípico" en el boxplot **no significa erróneo**: aquí son pulsos débiles (medidos más cortos), perfectamente reales.

### 6.5 Violínplot
📘 El **violínplot** combina una KDE (dibujada en espejo) con la información del boxplot. Muestra **dónde se concentran** los datos y si hay **varios peaks**.

### 6.6 ⚠️ El boxplot puede engañar
Generamos dos conjuntos **artificiales** de PRI: **A**, valores repartidos al azar entre 820 y 1180 µs (como un *jitter*), y **B**, dos peaks en 900 y 1100 µs (como un *stagger*). Mira sus boxplots… y luego sus violines.

In [ ]:
rng = np.random.default_rng(1)
conj_a = rng.uniform(820, 1180, 2000)                                    # tipo "jitter"
conj_b = np.r_[rng.normal(900, 25, 1000), rng.normal(1100, 25, 1000)]   # tipo "stagger"
demo = pd.DataFrame({"PRI (µs)": np.r_[conj_a, conj_b],
                     "conjunto": ["A: repartido (jitter)"] * 2000 + ["B: dos picos (stagger)"] * 2000})

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=demo, x="conjunto", y="PRI (µs)", ax=ax[0], width=0.4)
ax[0].set_title("Boxplot: ¿se ven distintos?")
sns.violinplot(data=demo, x="conjunto", y="PRI (µs)", ax=ax[1], inner="quartile", cut=0)
ax[1].set_title("Violín: la diferencia salta a la vista")
plt.tight_layout(); plt.show()

🎯 Los dos boxplots son casi idénticos (cajas de ≈ 900 a ≈ 1100, bigotes parecidos), pero el conjunto B **casi no tiene datos cerca de 1000** y el A los tiene repartidos parejo. Dos modulaciones de PRI **completamente distintas** con el mismo boxplot. Con **dos emisores mezclados** pasa lo mismo.

### 6.7 Ahora con los datos reales: la PRI del emisor
Tomamos los DTOA entre 700 y 1300 µs (descartamos huecos y pulsos perdidos) y usamos la función de 3 vistas de la caja de herramientas:

In [ ]:
def graficar_distribucion(serie, titulo="", unidad="", bins=60):
    """Histograma + KDE, boxplot y violín de una misma variable (3 vistas)."""
    x = pd.Series(serie).dropna()
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8), gridspec_kw={"width_ratios": [2.2, 1, 1]})
    sns.histplot(x, bins=bins, kde=True, ax=ax[0], color="#4C72B0")
    ax[0].axvline(x.mean(), color="#C44E52", ls="--", lw=1.5, label=f"media = {x.mean():.4g}")
    ax[0].axvline(x.median(), color="#55A868", ls="-", lw=1.5, label=f"mediana = {x.median():.4g}")
    ax[0].legend(fontsize=9); ax[0].set_title(f"Histograma + KDE · {titulo}"); ax[0].set_xlabel(unidad)
    sns.boxplot(y=x, ax=ax[1], color="#8172B2", width=0.4); ax[1].set_title("Boxplot"); ax[1].set_ylabel(unidad)
    sns.violinplot(y=x, ax=ax[2], color="#64B5CD", inner="quartile", cut=0); ax[2].set_title("Violín"); ax[2].set_ylabel(unidad)
    plt.tight_layout(); plt.show()

pri_principal = dtoa[(dtoa > 700) & (dtoa < 1300)]
graficar_distribucion(pri_principal, "DTOA entre 700 y 1300 µs", "µs", bins=120)
print(f"Media = {pri_principal.mean():.1f} µs   ·   pulsos con DTOA entre 950 y 1050 µs: {pri_principal.between(950, 1050).sum()}")

🎯 La **media (≈ 1000 µs) cae justo en el hueco** donde no hay ningún pulso. Reportar "PRI = 1000 µs" sería un **error de análisis**: el radar usa **dos** PRI (≈ 900 y ≈ 1100 µs). La **PRI media** (1000 µs) sí tiene sentido físico como dato *derivado* del stagger, pero **no** como valor de PRI.

**Regla:** primero **mira la forma** (histograma/violín), después elige el estadígrafo.

### 6.8 ECDF: leer percentiles directamente
📘 La **ECDF** (función de distribución acumulada empírica) muestra, para cada valor *x*, qué fracción de los datos es ≤ *x*. Para leer un percentil, se entra por el eje vertical.

In [ ]:
plt.figure(figsize=(10, 4))
sns.ecdfplot(df["FREQ_MHz"])
for p in [0.05, 0.5, 0.95]:
    v = df["FREQ_MHz"].quantile(p)
    plt.axhline(p, color="gray", ls=":", lw=1); plt.axvline(v, color="gray", ls=":", lw=1)
    plt.text(v, p + 0.03, f"P{int(p*100)} = {v:.2f}", fontsize=9)
plt.xlabel("RF (MHz)"); plt.ylabel("Fracción acumulada de pulsos")
plt.title("ECDF de la RF")
plt.show()

### 6.9 Comparar grupos: ¿mide igual el receptor a pulsos fuertes y débiles?
Separamos los pulsos del **haz principal** (AMP > −50 dBm) de los **débiles** (≤ −50 dBm) y comparamos su PW y su RF con violines lado a lado.

In [ ]:
df["NIVEL"] = np.where(df["AMP_dBm"] > -50, "Haz principal (> −50 dBm)", "Débiles (≤ −50 dBm)")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.violinplot(data=df, x="NIVEL", y="PW_us", ax=ax[0], inner="quartile", cut=0)
ax[0].set_title("PW según nivel de señal")
sns.violinplot(data=df, x="NIVEL", y="FREQ_MHz", ax=ax[1], inner="quartile", cut=0)
ax[1].set_title("RF según nivel de señal")
plt.tight_layout(); plt.show()

df.groupby("NIVEL")[["PW_us", "FREQ_MHz"]].agg(["median", "std"]).round(3)

📘 `groupby("columna")` separa la tabla en grupos y `agg([...])` calcula estadígrafos **para cada grupo**. Lo usaremos muchísimo (un grupo = un emisor).

🎯 Con señal débil el receptor mide el PW **más corto** y la RF con **más dispersión**. Consecuencia práctica: los parámetros del informe deben calcularse con estadígrafos robustos (mediana) o preferentemente con los pulsos de buena señal.

### ✏️ Tu turno 3
Haz un violín de `AOA_deg` por `NIVEL`. ¿Qué pasa con la precisión del ángulo de llegada en los pulsos débiles? ¿Qué implica para la **localización** del emisor?

In [ ]:
# ✏️ completa

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


```python
sns.violinplot(data=df, x="NIVEL", y="AOA_deg", inner="quartile", cut=0); plt.show()
df.groupby("NIVEL")["AOA_deg"].std()
```
La dispersión del AOA es **mucho mayor** con señal débil (varios grados vs ~1°). Para estimar la marcación del emisor conviene usar los pulsos del haz principal: un error de 3° a 100 km son ±5 km de incertidumbre lateral.


</details>

---
## 7. Forma de la distribución: asimetría

📘 El **coeficiente de asimetría** (*skewness*) resume hacia dónde se estira la distribución:

| Asimetría | Forma | Relación típica |
|---|---|---|
| ≈ 0 | Simétrica | media ≈ mediana |
| > 0 | Cola a la **derecha** | media > mediana |
| < 0 | Cola a la **izquierda** | media < mediana |

In [ ]:
from scipy import stats
for col in ["FREQ_MHz", "PW_us", "AMP_dBm"]:
    print(f"{col:9s} asimetría = {stats.skew(df[col]):6.2f}")

---
## 8. Intervalos de confianza (IC)

### 8.1 La idea
📘 La media de **nuestra** muestra no es exactamente el valor real del emisor: con otra captura obtendríamos una media un poco distinta. El **error estándar** cuantifica cuánto "baila" la media de una muestra a otra:

$$EE = \frac{s}{\sqrt{n}}$$

El **intervalo de confianza del 95 %** para la media es:

$$IC_{95\%} = \bar x \pm t_{0{,}975;\,n-1}\cdot\frac{s}{\sqrt n}\qquad(t \approx 1{,}96 \text{ para } n \text{ grande})$$

### 8.2 ¿Qué significa "95 % de confianza"? — Simulación
Supongamos un emisor cuya RF **real** es 2950,00 MHz y un receptor con error de medición σ = 0,8 MHz. Simulamos **100 capturas** de 20 pulsos cada una y calculamos el IC 95 % en cada captura:

In [ ]:
def ic95_media(serie, confianza=0.95):
    """Intervalo de confianza para la MEDIA usando la distribución t de Student."""
    x = pd.Series(serie).dropna().values
    n = len(x)
    ee = x.std(ddof=1) / np.sqrt(n)                       # error estándar
    t = stats.t.ppf(0.5 + confianza / 2, df=n - 1)       # ≈ 2,09 para n = 20
    return x.mean() - t * ee, x.mean() + t * ee

RF_REAL, SIGMA, N_PULSOS = 2950.0, 0.8, 20
rng = np.random.default_rng(7)
fallos = 0
plt.figure(figsize=(12, 6))
for i in range(100):
    muestra = rng.normal(RF_REAL, SIGMA, N_PULSOS)
    lo, hi = ic95_media(muestra)
    acierta = lo <= RF_REAL <= hi
    fallos += not acierta
    plt.plot([lo, hi], [i, i], color="#4C72B0" if acierta else "#C44E52", lw=2 if not acierta else 1.2)
    plt.plot(muestra.mean(), i, ".", color="k", ms=3)
plt.axvline(RF_REAL, color="k", ls="--", label="RF real (desconocida en la práctica)")
plt.xlabel("RF (MHz)"); plt.ylabel("Nº de captura simulada")
plt.title(f"100 intervalos de confianza del 95 %: {100 - fallos} contienen el valor real, {fallos} no (en rojo)")
plt.legend(); plt.show()

🎯 **Interpretación correcta:** el **método** produce intervalos que contienen el valor real en ~95 de cada 100 capturas. Con una sola captura no sabemos si la nuestra es de las "rojas", pero confiamos en el procedimiento.

| Frase | ¿Correcta? |
|---|---|
| "Con 95 % de confianza, la RF media del emisor está entre 2949,96 y 2950,02 MHz." | ✅ (forma habitual de reportarlo) |
| "El 95 % de los pulsos tiene una RF entre 2949,96 y 2950,02 MHz." | ❌ Eso sería un rango de datos (≈ P2,5–P97,5), mucho más ancho |
| "El emisor opera en el rango 2949,96 – 2950,02 MHz." | ❌ El IC habla de la **media**, no del rango de operación |
| "Si repito la captura, el 95 % de las veces la media caerá en este intervalo." | ❌ Lo que se repite es el **procedimiento**, no este intervalo fijo |

### 8.3 Efecto del tamaño de la muestra

In [ ]:
ns = np.array([5, 10, 20, 50, 100, 200, 500, 1000, 2000])
anchos = [2 * stats.t.ppf(0.975, n - 1) * SIGMA / np.sqrt(n) for n in ns]
plt.figure(figsize=(10, 4))
plt.plot(ns, anchos, "o-")
plt.xscale("log")
plt.xlabel("Nº de pulsos (escala log)"); plt.ylabel("Ancho del IC 95 % (MHz)")
plt.title("El IC se estrecha como 1/√n: 4 veces más pulsos → la mitad de ancho")
plt.grid(alpha=0.4); plt.show()

### 8.4 IC para la mediana: *bootstrap*
📘 Para la mediana no hay una fórmula simple. El **bootstrap** la resuelve "por fuerza bruta": se remuestrea la captura con reemplazo miles de veces, se calcula la mediana de cada remuestreo y se toman los percentiles 2,5 y 97,5 de esas medianas.

In [ ]:
def ic95_mediana_bootstrap(serie, n_boot=2000, confianza=0.95, semilla=0):
    """Intervalo de confianza para la MEDIANA por remuestreo (bootstrap)."""
    x = pd.Series(serie).dropna().values
    rng = np.random.default_rng(semilla)
    medianas = np.median(rng.choice(x, size=(n_boot, len(x)), replace=True), axis=1)
    a = (1 - confianza) / 2
    return np.quantile(medianas, a), np.quantile(medianas, 1 - a)

for col in ["FREQ_MHz", "PW_us"]:
    lo, hi = ic95_media(df[col]); blo, bhi = ic95_mediana_bootstrap(df[col])
    print(f"{col:9s} media {df[col].mean():9.3f}  IC95 [{lo:9.3f} ; {hi:9.3f}]   |   "
          f"mediana {df[col].median():9.3f}  IC95 [{blo:9.3f} ; {bhi:9.3f}]")

### 8.5 ⚠️ Tres advertencias del analista
1. **Precisión estadística ≠ exactitud del sensor.** Con 2000 pulsos el IC de la RF mide ±0,03 MHz, pero si el receptor tiene una exactitud de calibración de ±2 MHz, **ese** es el límite real. Repórtalos por separado: *"RF 2950,0 MHz (IC95 ± 0,03 MHz; exactitud del sensor ± 2 MHz)"*.
2. **Los pulsos no son totalmente independientes** (los de una misma iluminación comparten nivel de señal, condiciones de propagación…). El IC "de fórmula" es **optimista**.
3. **Un IC de una mezcla no tiene sentido.** Calcular el IC de la media de *todos* los DTOA o de una PRI con stagger produce un número preciso… de algo que no existe.

### 8.6 El IC correcto para la PRI de este emisor
Separamos los dos niveles del stagger y calculamos el IC de cada uno:

In [ ]:
filas = []
for nombre, (a, b) in {"PRI corta": (850, 950), "PRI larga": (1050, 1150)}.items():
    s = dtoa[dtoa.between(a, b)]
    lo, hi = ic95_media(s)
    filas.append({"Nivel": nombre, "n": len(s), "Media (µs)": s.mean(), "IC95 inf": lo, "IC95 sup": hi,
                  "Desv. est. (µs)": s.std(), "PRF (Hz)": 1e6 / s.mean()})
tabla_pri = pd.DataFrame(filas).round(3)
tabla_pri

In [ ]:
lo_mal, hi_mal = ic95_media(dtoa)
print(f"❌ IC de TODOS los DTOA: {dtoa.mean():,.0f} µs  [{lo_mal:,.0f} ; {hi_mal:,.0f}]  ← número sin significado físico")
print(f"✅ Stagger 2 posiciones: {tabla_pri.loc[0,'Media (µs)']:.2f} / {tabla_pri.loc[1,'Media (µs)']:.2f} µs · "
      f"periodo de trama {tabla_pri['Media (µs)'].sum():.2f} µs · PRI media {tabla_pri['Media (µs)'].mean():.2f} µs")

---
## 9. Valores atípicos: ¿error o información?

### 9.1 Dos reglas clásicas
| Regla | Criterio de "atípico" | Supone |
|---|---|---|
| **IQR (Tukey)** | < Q1 − 1,5·IQR  o  > Q3 + 1,5·IQR | Nada (robusta) |
| **Puntaje z** | $|z| = |x-\bar x|/s > 3$ | Distribución ≈ normal |

In [ ]:
def atipicos_iqr(serie, k=1.5):
    q1, q3 = serie.quantile([0.25, 0.75]); iqr = q3 - q1
    return (serie < q1 - k * iqr) | (serie > q3 + k * iqr)

def atipicos_z(serie, z=3):
    return ((serie - serie.mean()) / serie.std()).abs() > z

for col in ["FREQ_MHz", "PW_us", "DTOA_us"]:
    s = df[col].dropna()
    print(f"{col:9s} atípicos IQR: {atipicos_iqr(s).sum():5d}   ·   atípicos z>3: {atipicos_z(s).sum():5d}   (de {len(s)})")

In [ ]:
at = atipicos_iqr(df["FREQ_MHz"])
print("Amplitud mediana de los pulsos con RF 'atípica':", df.loc[at, "AMP_dBm"].median(), "dBm")
print("Amplitud mediana del resto                     :", df.loc[~at, "AMP_dBm"].median(), "dBm")

🎯 Los pulsos con RF "atípica" son, casi todos, **pulsos débiles** medidos con más error: no son errores, son mediciones menos precisas. Y en `DTOA_us`, la regla IQR marca como atípicos **todos los pulsos perdidos y todos los huecos de scan**, que son justamente la información para estimar el **periodo de exploración**.

### 9.2 ⚠️ Experimento: filtrar atípicos puede borrar un emisor completo
Imagina una captura con **dos** emisores: A (90 % de los pulsos, 2950 MHz) y B (10 %, 3100 MHz, quizás el más peligroso). Apliquemos la "limpieza" de atípicos por IQR sobre la RF:

In [ ]:
rng = np.random.default_rng(3)
mezcla = pd.Series(np.r_[rng.normal(2950, 0.8, 900), rng.normal(3100, 0.8, 100)], name="FREQ_MHz")
limpia = mezcla[~atipicos_iqr(mezcla)]
print(f"Pulsos del emisor B antes del filtro : {(mezcla > 3000).sum()}")
print(f"Pulsos del emisor B después          : {(limpia > 3000).sum()}   ← ¡desapareció!")

**Conclusiones para el analista**
* Un dato atípico es una **pregunta**, no una respuesta: investígalo antes de borrarlo.
* **Nunca** filtres atípicos sobre una captura con **varios emisores** mezclados: elimina a los minoritarios (separa primero, Labs 4–5).
* Filtrar por percentiles requiere cuidado: quedarse con los datos entre P50 y P65, por ejemplo, ¡**descarta el 85 %** de la captura!
* Para *resumir* un emisor ya separado, en lugar de borrar, usa **estadígrafos robustos** (mediana, IQR, P5–P95).

---
## 10. Producto: ficha estadística del emisor

In [ ]:
def resumen_estadistico(serie, nombre=None):
    """Tabla con los estadígrafos más usados en un informe de análisis."""
    x = pd.Series(serie).dropna()
    lo, hi = ic95_media(x)
    return pd.Series({
        "n": int(len(x)), "media": x.mean(), "IC95_media_inf": lo, "IC95_media_sup": hi,
        "mediana": x.median(), "moda (histograma)": moda_continua(x), "desv_estandar": x.std(ddof=1),
        "CV_%": 100 * x.std(ddof=1) / abs(x.mean()), "minimo": x.min(), "P5": x.quantile(0.05),
        "Q1 (P25)": x.quantile(0.25), "Q3 (P75)": x.quantile(0.75), "P95": x.quantile(0.95),
        "maximo": x.max(), "IQR": x.quantile(0.75) - x.quantile(0.25), "asimetria": stats.skew(x),
    }, name=nombre or getattr(serie, "name", "variable"))

fuertes = df[df["AMP_dBm"] > -50]                      # haz principal: mejores mediciones
ficha = pd.concat([
    resumen_estadistico(fuertes["FREQ_MHz"], "RF (MHz)"),
    resumen_estadistico(fuertes["PW_us"], "PW (µs)"),
    resumen_estadistico(fuertes["AOA_deg"], "AOA (°)"),
    resumen_estadistico(dtoa[dtoa.between(850, 950)], "PRI corta (µs)"),
    resumen_estadistico(dtoa[dtoa.between(1050, 1150)], "PRI larga (µs)"),
], axis=1).T
ficha.round(3)

In [ ]:
os.makedirs("resultados", exist_ok=True)
ficha.round(4).to_excel("resultados/D1_ficha_estadistica.xlsx")
print("Ficha guardada en resultados/D1_ficha_estadistica.xlsx ✔")

### ✏️ Tu turno 4 · Redacta el párrafo de parámetros para el informe
Con la ficha anterior, completa (doble clic):

> **RF:** ______ MHz (IC95 % ______ – ______ MHz); 90 % de los pulsos entre P5 = ______ y P95 = ______ MHz. Banda IEEE ___ / OTAN ___. RF fija.
> **PW:** mediana ______ µs (P5–P95: ______ – ______ µs).
> **PRI:** stagger de 2 posiciones, ______ / ______ µs (PRF ______ / ______ Hz); periodo de trama ______ µs.
> **AOA:** ______ ° ± ______ ° (1 desviación estándar).

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


Valores aproximados (pueden variar en el último decimal):

> **RF:** 2950,0 MHz (IC95 % ≈ 2949,96 – 2950,02); P5–P95 ≈ 2949,4 – 2950,6 MHz. Banda **S** (IEEE) / **E** (OTAN). RF fija.
> **PW:** mediana ≈ 1,20 µs (P5–P95 ≈ 1,17 – 1,22 µs).
> **PRI:** stagger 2 posiciones ≈ **900 / 1100 µs** (PRF ≈ 1111 / 909 Hz); trama 2000 µs; PRI media 1000 µs.
> **AOA:** ≈ 135° ± 1,3° (1 desv. est.).


</details>

---
## 11. Guía rápida de interpretación (para pegar en tu cuaderno)

### ¿Qué estadígrafo reporto?
| Situación | Valor central | Dispersión | Rango |
|---|---|---|---|
| Distribución simétrica, sin extremos (p. ej. RF fija) | Media (con IC 95 %) | Desv. estándar | P1–P99 o P5–P95 |
| Asimétrica o con pulsos extremos (PW, AOA con señal débil) | **Mediana** (IC bootstrap) | IQR | P5–P95 |
| **Varios peaks** (stagger, D&S, agilidad, mezcla) | **Separar** y reportar cada modo | Por modo | Por modo |
| Amplitud | No resumir con media: analizar **vs tiempo** (scan) | — | Máximo (haz principal) |

### ¿Qué gráfico uso?
| Gráfico | Úsalo para | Cuidado con |
|---|---|---|
| **Histograma** | Ver forma, peaks, valores discretos | El número de bins cambia la historia |
| **KDE** | Forma suavizada, comparar curvas | El ancho de banda puede fundir peaks; desborda límites |
| **Boxplot** | Comparar mediana/dispersión entre grupos | **Oculta la multimodalidad** |
| **Violín** | Comparar grupos **y** ver modos | Hereda los problemas de la KDE |
| **ECDF** | Leer percentiles exactos | Menos intuitivo al principio |
| **Dispersión vs tiempo** | Scan, cambios de modo, agilidad | Muchos puntos → usar `ms` pequeño |

### Preguntas de repaso
1. Un compañero informa "PRI = 1000 µs ± 0,5 µs (IC 95 %)" para este emisor. ¿Qué le dirías?
2. ¿Por qué el IC de la RF se hace más angosto con más pulsos, pero el rango P5–P95 no?
3. Un boxplot de la PW de un emisor muestra 40 "atípicos". ¿Los eliminas?

<details>
<summary><b>🔑 Ver solución</b> (inténtalo primero)</summary>


1. Que el número es preciso pero **engañoso**: la distribución es bimodal (stagger 900/1100 µs). Debe reportar los **dos niveles** (cada uno con su IC) y, si quiere, la PRI media 1000 µs **como dato derivado**.
2. El IC estima la **media** (incertidumbre ∝ 1/√n); el P5–P95 describe la **dispersión de los pulsos**, que es propia del emisor + receptor y no disminuye con más datos.
3. No sin investigar: si son pulsos débiles medidos peor, son reales; para el informe se usa la **mediana** y el rango P5–P95, o se calcula con los pulsos de buena señal.


</details>

---
### ✅ Fin del Laboratorio 2

**Antes de continuar**, guarda tu trabajo: *Archivo → Guardar una copia en Drive* (Colab) o *File → Save* (Jupyter).

➡️ **Siguiente:** **Laboratorio 3 · Caracterización de un emisor** (tipos de PRI, tipos de scan, agilidad en frecuencia y cambios de modo; 8 capturas, una por pareja).

<span style="font-size:11px;color:#7a7a7a">Curso de Guerra Electrónica · Análisis de Señales de Radar · Material de instrucción con datos sintéticos.</span>